In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pace_pic_validation.io.calcification_pic_processing import pic_from_ca

## i need volumes of filtrations to finish this


In [ ]:

alkalinity_results_path = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Research_Campaigns/Norbjørn /August 2026/data/BE20260729_salinity.csv'

df_alk = pd.read_csv(alkalinity_results_path)
print(df_alk.columns)

In [ ]:
alk_cols = [
    col for col in df_alk.columns
    if str(col).strip().startswith("Alk")
]


if not alk_cols:
    raise ValueError("No columns starting with 'Alk' were found.")

df_alk["alk_mean"] = df_alk[alk_cols].mean(axis=1)
df_alk["alk_std"] = df_alk[alk_cols].std(axis=1)

station = pd.to_numeric(df_alk['Station'], errors='raise')
keep_station = station.notna() & station.mod(1).eq(0)
df_alk = df_alk.loc[keep_station].copy()
df_alk['Station'] = station.loc[keep_station].astype('Int64')
df_alk['full_sample_id'] = df_alk['Cruise'].astype(str) + '_' + df_alk['Station'].astype(str) + '_' + df_alk['Sample ID'].astype(str)

df_alk_t0 = df_alk[df_alk["Time ID "] == 't0'].copy()
df_alk_t1 = df_alk[df_alk["Time ID "] == 't1'].copy()



 



In [ ]:
import matplotlib.pyplot as plt

t0_for_diff = df_alk_t0[['full_sample_id', 'alk_mean', 'alk_std']].rename(
    columns={'alk_mean': 'alk_mean_t0', 'alk_std': 'alk_std_t0'}
)
t1_for_diff = df_alk_t1[['full_sample_id', 'alk_mean', 'alk_std']].rename(
    columns={'alk_mean': 'alk_mean_t1', 'alk_std': 'alk_std_t1'}
)

alk_change = t0_for_diff.merge(
    t1_for_diff,
    on='full_sample_id',
    how='inner',
    validate='one_to_one',
)
if alk_change.empty:
    raise ValueError("No matching full_sample_id values were found between t0 and t1.")

unmatched_t0 = set(t0_for_diff['full_sample_id']) - set(t1_for_diff['full_sample_id'])
unmatched_t1 = set(t1_for_diff['full_sample_id']) - set(t0_for_diff['full_sample_id'])
print(f"Unmatched t0 IDs: {len(unmatched_t0)}; unmatched t1 IDs: {len(unmatched_t1)}")

alk_change['alk_difference_t0_minus_t1'] = alk_change['alk_mean_t0'] - alk_change['alk_mean_t1']
alk_change['alk_difference_std'] = np.sqrt(
    alk_change['alk_std_t0'] ** 2 + alk_change['alk_std_t1'] ** 2
)

plot_values = alk_change[['alk_difference_t0_minus_t1', 'alk_difference_std']].to_numpy()
if not np.isfinite(plot_values).all():
    bad_ids = alk_change.loc[~np.isfinite(plot_values).all(axis=1), 'full_sample_id'].tolist()
    raise ValueError(f"Missing or non-finite alkalinity values/SDs for matched IDs: {bad_ids}")

fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(alk_change)), 5))
ax.bar(
    alk_change['full_sample_id'],
    alk_change['alk_difference_t0_minus_t1'],
    yerr=alk_change['alk_difference_std'],
    capsize=4,
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Sample ID')
ax.set_ylabel('Alkalinity difference (t0 - t1)')
ax.tick_params(axis='x', labelrotation=90)
fig.tight_layout()
plt.show()
